# Merge csv: clinical, demographic...

In [ ]:
import os
from pathlib import Path
import pandas as pd

from ppmi_utils import (
    load_ppmi_csvs,
    merge_ppmi_tables,
    plot_hist,
    visit_to_event_id,
    event_id_to_visit,
)

ROOT_DIR = Path(os.getcwd()).resolve().parents[1]
PPMI_CLINICAL = ROOT_DIR / "csv_dir" / "PPMI_CLINICAL"
PPMI_OTHERS = ROOT_DIR / "csv_dir" / "PPMI_OTHERS"

In [ ]:
ppmi_data = load_ppmi_csvs(PPMI_CLINICAL)
print(f"Available datasets: {list(ppmi_data.keys())}")
# for name, df in ppmi_data.items():
#     print(name, ":", set(["PATNO"]).issubset(df.columns))
#     print(name, ":", set(["PATNO","EVENT_ID"]).issubset(df.columns))


In [ ]:
merged_tabular_df = merge_ppmi_tables(ppmi_data)
print("Merged DataFrame shape:", merged_tabular_df.shape)
print("Merged DataFrame columns:", merged_tabular_df.columns.tolist())

# Add idaSearch (image info)

In [ ]:
ida_df = ppmi_data["idaSearch"].copy()
ida_df = ida_df.rename(columns={"SUBJECT ID": "PATNO"})
ida_df["EVENT_ID"] = ida_df["VISIT"].map(visit_to_event_id)
merged_tabular_df["VISIT"] = merged_tabular_df["EVENT_ID"].map(event_id_to_visit)
# print(ida_df[["VISIT", "EVENT_ID"]].head())
# ida_df["EVENT_ID"].value_counts()


In [ ]:
# Step 3️⃣: Merge the DataFrames
ida_df["PATNO"] = ida_df["PATNO"].astype(str)
merged_tabular_df["PATNO"] = merged_tabular_df["PATNO"].astype(str)
final_df = pd.merge(
    merged_tabular_df,
    ida_df,
    how="outer",  # or "inner" if clinical_df['PATNO'] = clinical_df['PATNO'].astype(str)only want matches
    on=["PATNO", "EVENT_ID"],
)
print("Final merged DataFrame shape:", final_df.shape)

In [ ]:
patient_ex = final_df[
    (final_df["PATNO"] == "100445") & (final_df["EVENT_ID"] == "BL")
].copy()
print(patient_ex.shape)
# print(patient_ex)
# patient_ex.to_csv("patient_100445_exploration.csv", index=False)

# Basic info

Sex, diagnosis, age

# Age at baseline

In [ ]:
# age distribution at baseline
baseline_df = final_df[final_df["EVENT_ID"] == "BL"]
print("Baseline age distribution:")
print(baseline_df["AGE_AT_VISIT"].describe())
plot_hist(baseline_df, "AGE_AT_VISIT")

#
visit_counts = final_df.groupby("PATNO")["EVENT_ID"].nunique()
print(f"Visit counts per subject: {visit_counts}")